In [1]:
import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 192) / Test: (90067, 191)


In [2]:
# feature_refinement 재적용 + 피처 준비
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)

print('피처 정제 완료')

피처 정제 완료


In [3]:
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

# CatBoost용
cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

# LightGBM용
X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

# XGBoost용: OrdinalEncoder + 중앙값 대체
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
X_xgb[cat_features]      = enc.fit_transform(X[cat_features])
X_test_xgb[cat_features] = enc.transform(X_test[cat_features])
num_features = [c for c in feat_cols if c not in cat_features]
medians = X_xgb[num_features].median()
X_xgb[num_features]      = X_xgb[num_features].fillna(medians)
X_test_xgb[num_features] = X_test_xgb[num_features].fillna(medians)

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

N_SPLITS = 5
SEED     = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

피처: 197개 / 범주형: 21개
scale_pos_weight: 2.87


In [4]:
# XGBoost OPtuna
def xgb_objective(trial):
    params = {
        'objective':        'binary:logistic',
        'eval_metric':      'auc',
        'tree_method':      'hist',
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1000),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 2.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 1.0),
        'scale_pos_weight': spw,
        'random_state':     SEED,
        'verbosity':        0,
        'early_stopping_rounds': 30,
    }

    skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    fold_aucs = []

    for fold, (tr_idx, val_idx) in enumerate(skf3.split(X_xgb, y)):
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_xgb.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X_xgb.iloc[val_idx], y.iloc[val_idx])],
            verbose=False
        )
        val_pred = model.predict_proba(X_xgb.iloc[val_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y.iloc[val_idx], val_pred))

        trial.report(np.mean(fold_aucs), fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_aucs)

print('XGBoost Objective 정의 완료')

XGBoost Objective 정의 완료


In [5]:
sampler   = optuna.samplers.TPESampler(seed=SEED)
pruner    = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
xgb_study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)

def print_callback(study, trial):
    if trial.value is None:
        return
    print(f'Trial {trial.number:3d} | AUC: {trial.value:.5f} | Best: {study.best_value:.5f} '
          f'| depth={trial.params.get("max_depth")}, '
          f'lr={trial.params.get("learning_rate"):.4f}, '
          f'iter={trial.params.get("n_estimators")}')

xgb_study.optimize(
    xgb_objective,
    n_trials=50,
    show_progress_bar=False,
    callbacks=[print_callback],
)

print(f'\n✅ 최적 AUC (3-fold): {xgb_study.best_value:.5f}')
print('최적 파라미터:')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

Trial   0 | AUC: 0.73922 | Best: 0.73922 | depth=5, lr=0.0893, iter=813
Trial   1 | AUC: 0.73909 | Best: 0.73922 | depth=7, lr=0.0105, iter=979
Trial   2 | AUC: 0.73958 | Best: 0.73958 | depth=5, lr=0.0196, iter=728
Trial   3 | AUC: 0.73935 | Best: 0.73958 | depth=6, lr=0.0391, iter=332
Trial   4 | AUC: 0.73920 | Best: 0.73958 | depth=4, lr=0.0125, iter=779
Trial   5 | AUC: 0.73947 | Best: 0.73958 | depth=6, lr=0.0205, iter=664
Trial   6 | AUC: 0.73917 | Best: 0.73958 | depth=6, lr=0.0835, iter=362
Trial   7 | AUC: 0.73966 | Best: 0.73966 | depth=5, lr=0.0191, iter=680
Trial   8 | AUC: 0.73943 | Best: 0.73966 | depth=3, lr=0.0654, iter=795
Trial   9 | AUC: 0.73954 | Best: 0.73966 | depth=6, lr=0.0214, iter=344
Trial  10 | AUC: 0.73882 | Best: 0.73966 | depth=8, lr=0.0409, iter=521
Trial  11 | AUC: 0.73952 | Best: 0.73966 | depth=4, lr=0.0192, iter=608
Trial  12 | AUC: 0.73964 | Best: 0.73966 | depth=5, lr=0.0160, iter=674
Trial  13 | AUC: 0.73831 | Best: 0.73966 | depth=4, lr=0.0141, i

In [6]:
# 최적 파라미터로 모델 5-fold 최종 학습 + 앙상블
# 최적 XGBoost 파라미터
best_xgb_params = xgb_study.best_params
best_xgb_params.update({
    'objective':    'binary:logistic',
    'eval_metric':  'auc',
    'tree_method':  'hist',
    'scale_pos_weight': spw,
    'random_state': SEED,
    'verbosity':    0,
    'early_stopping_rounds': 50,
})

# CatBoost / LightGBM 파라미터 (이전 최적값)
cat_params = {
    'iterations': 794, 'learning_rate': 0.030094391673729362,
    'depth': 7, 'l2_leaf_reg': 7.433949857398995,
    'bagging_temperature': 0.3980358270898108,
    'random_strength': 1.3075975210328405, 'border_count': 108,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'scale_pos_weight': spw, 'random_seed': SEED,
    'early_stopping_rounds': 50, 'verbose': False, 'use_best_model': True,
}
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.02030926523583015,
    'n_estimators': 956, 'subsample': 0.9916893928795039,
    'colsample_bytree': 0.8760988294276582,
    'reg_alpha': 0.5158539052029842, 'reg_lambda': 0.05557192411355649,
    'min_child_samples': 98,
    'scale_pos_weight': spw, 'random_state': SEED, 'verbose': -1,
}

oof_cat  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr  = X.iloc[tr_idx].reset_index(drop=True)
    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_tr  = y.iloc[tr_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    # CatBoost
    cb = CatBoostClassifier(**cat_params)
    cb.fit(Pool(X_tr, y_tr, cat_features=cat_feature_indices),
           eval_set=Pool(X_val, y_val, cat_features=cat_feature_indices))
    oof_cat[val_idx]  = cb.predict_proba(Pool(X_val, cat_features=cat_feature_indices))[:, 1]
    test_cat         += cb.predict_proba(Pool(X_test, cat_features=cat_feature_indices))[:, 1] / N_SPLITS

    # LightGBM
    lb = lgb.LGBMClassifier(**lgb_params)
    lb.fit(X_lgb.iloc[tr_idx], y_tr,
           eval_set=[(X_lgb.iloc[val_idx], y_val)],
           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = lb.predict_proba(X_lgb.iloc[val_idx])[:, 1]
    test_lgb         += lb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

    # XGBoost (튜닝된 파라미터)
    xb = xgb.XGBClassifier(**best_xgb_params)
    xb.fit(X_xgb.iloc[tr_idx], y_tr,
           eval_set=[(X_xgb.iloc[val_idx], y_val)],
           verbose=False)
    oof_xgb[val_idx]  = xb.predict_proba(X_xgb.iloc[val_idx])[:, 1]
    test_xgb         += xb.predict_proba(X_test_xgb)[:, 1] / N_SPLITS

    cat_auc = roc_auc_score(y_val, oof_cat[val_idx])
    lgb_auc = roc_auc_score(y_val, oof_lgb[val_idx])
    xgb_auc = roc_auc_score(y_val, oof_xgb[val_idx])
    print(f'Fold {fold+1} | CAT: {cat_auc:.5f} | LGB: {lgb_auc:.5f} | XGB: {xgb_auc:.5f}')

print(f'\n✅ CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}')
print(f'✅ LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}')
print(f'✅ XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}  (튜닝 전: 0.73946)')

Fold 1 | CAT: 0.73848 | LGB: 0.73788 | XGB: 0.73833
Fold 2 | CAT: 0.74320 | LGB: 0.74207 | XGB: 0.74283
Fold 3 | CAT: 0.74083 | LGB: 0.74041 | XGB: 0.73989
Fold 4 | CAT: 0.73832 | LGB: 0.73825 | XGB: 0.73815
Fold 5 | CAT: 0.74077 | LGB: 0.74038 | XGB: 0.74066

✅ CatBoost OOF AUC : 0.74031
✅ LightGBM OOF AUC : 0.73978
✅ XGBoost  OOF AUC : 0.73997  (튜닝 전: 0.73946)


In [7]:
# 소프트 보팅 가중치 탐색 (0.05 단위 세밀화)
best_auc_vote, best_w = 0.0, (0.5, 0.3, 0.2)
results = []

# 0.05 단위 세밀 탐색
for w1 in np.arange(0.0, 1.05, 0.05):
    for w2 in np.arange(0.0, 1.05 - w1, 0.05):
        w3 = round(1 - w1 - w2, 2)
        if w3 < 0:
            continue
        blended = oof_cat * w1 + oof_lgb * w2 + oof_xgb * w3
        auc = roc_auc_score(y, blended)
        results.append((round(w1,2), round(w2,2), round(w3,2), auc))
        if auc > best_auc_vote:
            best_auc_vote = auc
            best_w = (w1, w2, w3)

results_df = pd.DataFrame(results, columns=['w_cat','w_lgb','w_xgb','oof_auc'])
print('=== 소프트 보팅 상위 10개 ===')
display(results_df.sort_values('oof_auc', ascending=False).head(10))
print(f'\n✅ 소프트 보팅 최적 AUC : {best_auc_vote:.5f}')
print(f'   최적 가중치: CAT={best_w[0]:.2f}, LGB={best_w[1]:.2f}, XGB={best_w[2]:.2f}')
print(f'   이전 최고 (2모델)    : 0.74041')
print(f'   개선폭               : {best_auc_vote - 0.74041:+.5f}')

=== 소프트 보팅 상위 10개 ===


,w_cat,w_lgb,w_xgb,oof_auc
189,0.60,0.15,0.25,0.740455
188,0.60,0.10,0.30,0.740453
197,0.65,0.10,0.25,0.740453
198,0.65,0.15,0.20,0.740452
179,0.55,0.15,0.30,0.740452
190,0.60,0.20,0.20,0.740452
180,0.55,0.20,0.25,0.740451
178,0.55,0.10,0.35,0.740449
196,0.65,0.05,0.30,0.740448
199,0.65,0.20,0.15,0.740448



✅ 소프트 보팅 최적 AUC : 0.74046
   최적 가중치: CAT=0.60, LGB=0.15, XGB=0.25
   이전 최고 (2모델)    : 0.74041
   개선폭               : +0.00005


In [8]:
# 제출 파일 생성
test_blended = test_cat * best_w[0] + test_lgb * best_w[1] + test_xgb * best_w[2]

sample_sub = pd.read_csv('/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw/sample_submission.csv')
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_blended
submission.to_csv('submission_xgb_optuna_ensemble.csv', index=False, encoding='utf-8-sig')

print('저장 완료: submission_xgb_optuna_ensemble.csv')
print(f'\n최종 요약')
print(f'  CatBoost  : {roc_auc_score(y, oof_cat):.5f}')
print(f'  LightGBM  : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  XGBoost   : {roc_auc_score(y, oof_xgb):.5f}  (튜닝 전: 0.73946)')
print(f'  앙상블     : {best_auc_vote:.5f}  (w_cat={best_w[0]:.2f}, w_lgb={best_w[1]:.2f}, w_xgb={best_w[2]:.2f})')
print(f'  이전 최고  : 0.74041')
print(f'  총 개선폭  : {best_auc_vote - 0.74008:+.5f}')
display(submission.head())

저장 완료: submission_xgb_optuna_ensemble.csv

최종 요약
  CatBoost  : 0.74031
  LightGBM  : 0.73978
  XGBoost   : 0.73997  (튜닝 전: 0.73946)
  앙상블     : 0.74046  (w_cat=0.60, w_lgb=0.15, w_xgb=0.25)
  이전 최고  : 0.74041
  총 개선폭  : +0.00038


,ID,probability
0,TEST_00000,0.002560
1,TEST_00001,0.015338
2,TEST_00002,0.342412
3,TEST_00003,0.247179
4,TEST_00004,0.727075
